# Homework 06: Data Preprocessing

Applies the three cleaning functions in `src/cleaning.py` to a small synthetic dataset with
missing values, saves the cleaned result, and compares it against the original.

## Setup: Generate the Sample Dataset
Same synthetic dataset as the starter, with missing values built in on purpose.

In [1]:
import os
import numpy as np
import pandas as pd

raw_dir = 'data/raw'
processed_dir = 'data/processed'
os.makedirs(raw_dir, exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)

data = {
    'age': [34, 45, 29, 50, 38, np.nan, 41],
    'income': [55000, np.nan, 42000, 58000, np.nan, np.nan, 49000],
    'score': [0.82, 0.91, np.nan, 0.76, 0.88, 0.65, 0.79],
    'zipcode': ['90210', '10001', '60614', '94103', '73301', '12345', '94105'],
    'city': ['Beverly', 'New York', 'Chicago', 'SF', 'Austin', 'Unknown', 'San Francisco'],
    'extra_data': [np.nan, 42, np.nan, np.nan, np.nan, 5, np.nan],
}
df = pd.DataFrame(data)

csv_path = os.path.join(raw_dir, 'sample_data.csv')
df.to_csv(csv_path, index=False)
print(f'Sample dataset saved to {csv_path}')

Sample dataset saved to data/raw\sample_data.csv


## Load Raw Dataset

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from src import cleaning

df_raw = pd.read_csv('data/raw/sample_data.csv')
print('Missing values per column:')
print(df_raw.isna().sum())
df_raw

Missing values per column:
age           1
income        3
score         1
zipcode       0
city          0
extra_data    5
dtype: int64


,age,income,score,zipcode,city,extra_data
0,34.0,55000.0,0.82,90210,Beverly,NaN
1,45.0,NaN,0.91,10001,New York,42.0
2,29.0,42000.0,NaN,60614,Chicago,NaN
3,50.0,58000.0,0.76,94103,SF,NaN
4,38.0,NaN,0.88,73301,Austin,NaN
5,NaN,NaN,0.65,12345,Unknown,5.0
6,41.0,49000.0,0.79,94105,San Francisco,NaN


## Apply Cleaning Functions

Order matters here. `extra_data` is missing in 5 of 7 rows (71%), well past a reasonable
threshold to impute, so it gets dropped first with `drop_missing`. `age`, `income`, and `score`
have only one or two gaps each, so those get filled with the column median rather than dropped.
Then `income` and `score` get min-max normalized, since they're on very different scales
(tens of thousands vs. a 0-1 score) and a later model would otherwise let `income` dominate
purely because of its units.

In [3]:
df_clean = cleaning.drop_missing(df_raw, threshold=0.5)
print('Columns after drop_missing:', list(df_clean.columns))

df_clean = cleaning.fill_missing_median(df_clean, columns=['age', 'income', 'score'])
print('Missing values after fill_missing_median:')
print(df_clean.isna().sum())

df_clean = cleaning.normalize_data(df_clean, columns=['income', 'score'])
df_clean

Columns after drop_missing: ['age', 'income', 'score', 'zipcode', 'city']
Missing values after fill_missing_median:
age        0
income     0
score      0
zipcode    0
city       0
dtype: int64


,age,income,score,zipcode,city
0,34.0,0.8125,0.653846,90210,Beverly
1,45.0,0.6250,1.000000,10001,New York
2,29.0,0.0000,0.596154,60614,Chicago
3,50.0,1.0000,0.423077,94103,SF
4,38.0,0.6250,0.884615,73301,Austin
5,39.5,0.6250,0.000000,12345,Unknown
6,41.0,0.4375,0.538462,94105,San Francisco


## Compare Original vs Cleaned

In [4]:
comparison = pd.DataFrame({
    'original_shape': [df_raw.shape],
    'cleaned_shape': [df_clean.shape],
    'original_missing_total': [int(df_raw.isna().sum().sum())],
    'cleaned_missing_total': [int(df_clean.isna().sum().sum())],
})
comparison

,original_shape,cleaned_shape,original_missing_total,cleaned_missing_total
0,"(7, 6)","(7, 5)",10,0


## Save Cleaned Dataset

In [5]:
out_path = os.path.join(processed_dir, 'sample_data_cleaned.csv')
df_clean.to_csv(out_path, index=False)
print('Saved', out_path)

Saved data/processed\sample_data_cleaned.csv


## Assumptions and Notes

- A column missing more than half its values (`extra_data`, at 71%) is dropped rather than
  imputed. Filling that much of a column with its median would mostly be inventing data.
- Median, not mean, fills the remaining numeric gaps in `age`, `income`, and `score`, since
  median is less sensitive to the one or two outliers a small sample like this can have.
- `zipcode` and `city` are left untouched. They have no missing values here, and filling a
  missing zip code or city with a median doesn't mean anything the way it does for a number.
- Normalizing after filling, not before, matters: normalizing first would let the still-missing
  values distort the min/max the scaling is based on.
- Tradeoff worth calling out: dropping `extra_data` entirely means whatever signal was in that
  5-of-7-missing column is gone for good. With more data, a missingness indicator column
  (`extra_data_was_missing`) would preserve some of that information instead of just discarding it.